In [1]:
import yfinance as yf 
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
def check_type(data):
    for i in range(0,len(data)):
        if(data['prompt'].iloc[i]==1):
            return 1
        elif(data['prompt'].iloc[i]==-1):
            return 0
def returns(data):
    if(check_type(data)==1):
        returns_p=[]
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                for j in range(i+1,len(data)):
                    if(data['prompt'].iloc[j]==-1):
                        returns_p.append((data['Close'].iloc[j]-data['Close'].iloc[i])*100/data['Close'].iloc[i])
                        i=j
                        break
        returns_p=np.array(returns_p)
        return returns_p
    elif(check_type(data)==0):
        returns_p=[]
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                for j in range(i+1,len(data)):
                    if(data['prompt'].iloc[j]==1):
                        returns_p.append((data['Close'].iloc[i]-data['Close'].iloc[j])*100/data['Close'].iloc[j])
                        i=j
                        break
        returns_p=np.array(returns_p)
        return returns_p
def no_trades(data):
    return(len(returns(data)))
    
def MaxDrawDown(data):
    
#     max_peak_till_now = data["Close"].cummax()
#     drawdown = (data["Close"] - max_peak_till_now)/max_peak_till_now
#     return drawdown.min()*100
    final=np.zeros(no_trades(data))
    flag=0
    if(check_type(data)==1):
       
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                 for j in range(i+1,len(data)):
                         if(data['prompt'].iloc[j]==-1):
                                max_peak_till_now = data["Close"].iloc[i:j+1].cummax()
                                drawdown = (data["Close"].iloc[i:j+1] - max_peak_till_now)/max_peak_till_now
                                final[flag]=drawdown.min()*100
                                flag=flag+1
                                break
                                
    elif(check_type(data)==0):
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                 for j in range(i+1,len(data)):
                         if(data['prompt'].iloc[j]==1):
                                min_low_till_now = data["Close"].iloc[i:j+1].cummin()
                                drawdown = (min_low_till_now-data["Close"].iloc[i:j+1])/min_low_till_now
                                final[flag]=drawdown.min()*100
                                flag=flag+1
                                break
    return final.min()
                        
                
            
            

            
def SharpeRatio(data):
    
    rfr = 0
    return(252 * (returns(data).mean() - rfr )/(np.sqrt(252) * returns(data).std()))

def portfolio_value(data):
    capital=10000000
 
    if(check_type(data)==0):
        for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==-1):
                for j in range(0,len(data)):
                    if(data['prompt'].iloc[j]==1):
                        capital=(capital//data['Close'].iloc[j])*(data['Close'].iloc[i]) +capital%(data['Close'].iloc[j])
                        break
                        
    elif(check_type(data)==1):
           for i in range(0,len(data)):
            if(data['prompt'].iloc[i]==1):
                for j in range(0,len(data)):
                    if(data['prompt'].iloc[j]==-1):
                        capital=(capital//data['Close'].iloc[i])*(data['Close'].iloc[j]) +capital%(data['Close'].iloc[i])
                        break
        
    return capital
                        
        
data=yf.download('^NSEI',start='2018-01-01',end='2024-01-01')

data['macd']=data['Close'].ewm(span=12,adjust="False").mean()-data['Close'].ewm(span=26,adjust='False').mean()
data['signal']=data['macd'].ewm(span=9,adjust='False').mean()
data['prompt']=np.zeros(len(data['Close']))
for i in range(0,len(data['Close'])-1):
    if(data['macd'].iloc[i]-data['signal'].iloc[i]<0 and data['macd'].iloc[i+1]-data['signal'].iloc[i+1]>=0):
        data['prompt'].iloc[i]=1
    elif(data['macd'].iloc[i]-data['signal'].iloc[i]>0 and data['macd'].iloc[i+1]-data['signal'].iloc[i+1]<=0):  
        data['prompt'].iloc[i]=-1
        
else:
    data['prompt'].iloc[i]=0
      
        


print(data.to_string())
flag=0
for i in range(0,len(data)):
    if(data['prompt'].iloc[i]==1):
        flag=flag+1
    elif(data["prompt"].iloc[i]==-1):
        flag=flag-1
if(flag==1):
    data['prompt'].iloc[-1]=-1
elif(flag==-1):
    data['prompt'].iloc[-1]=1
print('THE NUMBER OF TRADES TAKEN IS:',no_trades(data))
print('RETURNS ON EVERY TRADE IS:',returns(data))
print('THE PORTFOLIO VALUE IS:',portfolio_value(data))
print('THE MAX DRAWDOWN IS:',MaxDrawDown(data),'%')
print('THE SHARPE RATIO IS:',SharpeRatio(data))

[*********************100%%**********************]  1 of 1 completed

1 Failed download:
['^NSEI']: ConnectionError(MaxRetryError("HTTPSConnectionPool(host='fc.yahoo.com', port=443): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x000001BD13CBCF10>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed'))"))


NameError: name 'i' is not defined